In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

### Pre-processing

#### 1. Converting to lowercase

In [7]:
df["review"] = df["review"].str.lower()

#### 2. Removing the URLs

In [8]:
import re

# sample_text = "abc is the word, abc" # abc -> xyz
# new_text = re.sub("abc", "xyz", sample_text)
# print(new_text)

In [9]:
def remove_urls(text):
    re.sub(r"http\S+", "", text)  # (pattern, replacement, string) ex- https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

#### 4. Removing HTML

In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text) # .*? => 0 or more chzra
    return text

df["review"] = df["review"].apply(remove_html)

#### 3. Remove punctuations

In [11]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # exclude A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

#### 5. Removing the Stopwords

In [12]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 

In [14]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)
# tokens

In [15]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
           text = text.replace(word, "")
        
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [16]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti filming technique un...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


#### 6. Stemming

In [17]:
# running => run, played => play
# Porter Stemming
from nltk.stem import PorterStemmer

In [18]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [19]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti film techniqu unssum oldtim...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


#### 7. Encoding

In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [21]:
y = df["sentiment"]

#### 8. Vectorization

In [22]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti film techniqu unssum oldtim...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000) # we are only using 1st 5000 data, all 50000 data will create a huge dimentional vector

X = tf.fit_transform(df["review"])

### Dataset & Dataloaders

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
X_train.shape
X_test.shape

(9917, 5000)

In [28]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [29]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [30]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)

In [31]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

### Build our RNN

In [34]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):  # input size = size of X, num_layers = how many RNN layers, hidden_size = hidden state size
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True) # batch_first=True => batch_size is given 1st as RNN function parameter
        
        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1) # input = hidden_size, output = 1 (Many to One RNN architecture for sentiment analysis)
        
    def forward(self, x):
        # optional => shape(num of layers, batch size, hidden size) 
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)  # By default, h0 is initialized as 0
        
        out, _ = self.rnn(x, h0)
        # returned value from RNN layer:
        # 1st value = hidden state of all the timesteps => stored in "out" => (batch_size, seq_len, hidden_size)
        # 2nd value = final hidden state of last timestep => is not stored
        
        out = self.fc(out[:, -1, :]) # we want the last timestep for all batch and hidden states, so seq_len = -1
        return out

In [35]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss() # Binary cross entropy loss(only 2 outputs)
optimizer = optim.Adam(model.parameters())

### Train the RNN

In [38]:
# unsqueeze => adds 1 dimension to data, and squeeze => removes 1 dimension

epochs = 10

for epoch in range(epochs):
    model.train()
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        xb = xb.unsqueeze(1) # add singleton direction (convert 2D => 3D input), because RNN expects 3D input
        
        outputs = model(xb) # output(returns 2D)
        
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => 1D probabilistic output
        
        loss = criterion(outputs, yb)  # loss compute 
        loss.backward() # back prop.
        optimizer.step() # weights update
        
    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        

epoch = 1/10 and loss = 0.09072955697774887
epoch = 2/10 and loss = 0.33962807059288025
epoch = 3/10 and loss = 0.24675951898097992
epoch = 4/10 and loss = 0.17056193947792053
epoch = 5/10 and loss = 0.2600853145122528
epoch = 6/10 and loss = 0.19273847341537476
epoch = 7/10 and loss = 0.21872355043888092
epoch = 8/10 and loss = 0.24600081145763397
epoch = 9/10 and loss = 0.21633003652095795
epoch = 10/10 and loss = 0.1718326359987259


### Evaluate

In [39]:
# Evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        
        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
        
    print(f"accuracy = {correct_vals/total_vals*100}")

accuracy = 85.44922859735807
